In [1]:
from nvidia.dali.pipeline import experimental, pipeline_def
from nvidia.dali.types import DALIDataType
from nvidia.dali import fn
from nvidia.dali import types
from nvidia.dali import tensors

from nvidia.dali.data_node import DataNode as _DataNode

from functools import wraps

import numpy as np

# dynamic and static magnitude
# shapes
# negation of the range
# big decorator

In [22]:
@experimental.pipeline_def(enable_conditionals=True, seed=42)
def pipeline():
    sample_idx = fn.external_source(
        source=lambda sample_info: np.array(sample_info.idx_in_batch),
        batch=False)
    stacked = fn.cast(fn.stack(sample_idx, sample_idx), dtype=types.FLOAT)
    if sample_idx < 2:
        m = fn.transforms.translation(offset=2 * stacked)
    else:
        m = fn.transforms.translation(offset=2 * stacked + 1)
    return sample_idx, m

In [23]:
# @pipeline_def
# def pipeline():
#     magnitudes = fn.random.uniform(range=[0, 5 - 1], dtype=types.INT32)
#     return magnitudes, fn.stack(np.array(0, dtype=np.int32), magnitudes)

In [24]:
p = pipeline(batch_size=5, num_threads=4, device_id=0)

In [25]:
p.build()

In [27]:
p.save_graph_to_dot_file("toomuchsplit.dot")

In [26]:
p.run()

RuntimeError: Critical error in pipeline:
Error when executing CPU operator conditional__Split, instance name: "__Split_22", encountered:
[/home/ktokarski/DALI/dali/pipeline/operator/builtin/split.cc:29] Assert on "input.num_samples() == predicate.num_samples()" failed: Split description must cover whole input, got 3 input samples and 5 elements denoting the split.
Stacktrace (11 entries):
[frame 0]: /home/ktokarski/DALI/build/dali/python/nvidia/dali/libdali.so(+0xca832) [0x7f3653728832]
[frame 1]: /home/ktokarski/DALI/build/dali/python/nvidia/dali/libdali.so(+0x198672) [0x7f36537f6672]
[frame 2]: /home/ktokarski/DALI/build/dali/python/nvidia/dali/libdali.so(+0x16df9b) [0x7f36537cbf9b]
[frame 3]: /home/ktokarski/DALI/build/dali/python/nvidia/dali/libdali.so(dali::Executor<dali::AOT_WS_Policy<dali::UniformQueuePolicy>, dali::UniformQueuePolicy>::RunHelper(dali::OpNode&, dali::Workspace&, unsigned long)+0x5d1) [0x7f36537a34f1]
[frame 4]: /home/ktokarski/DALI/build/dali/python/nvidia/dali

In [2]:
# @experimental.pipeline_def(enable_conditionals=True, seed=42)
# def pipeline():
#     r = fn.random.uniform(range=[0, 4], dtype=types.INT32)
#     s = np.array([np.full((2,), i, dtype=np.float32) for i in range(5)])
#     s = types.Constant(s)
#     if r <= 2:
#         t = fn.transforms.translation(offset=s[r])
#     else:
#         t = fn.transforms.translation(offset=s[r])
#     return r, s, t

In [3]:
# p = pipeline(batch_size=3, num_threads=4, device_id=0)

In [4]:
# p.build()

In [5]:
# p.run()

In [6]:
data_path = "/home/ktokarski/DALI_extra/db/single/jpeg"

import os
from PIL import Image, ImageDraw
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec

def display_imgs(outputs, columns=2, captions=None, cpu=True):
    rows = (len(outputs) + columns - 1) // columns
    fig = plt.figure()
    fig.set_size_inches(16, 6 * rows)
    gs = gridspec.GridSpec(rows, columns)
    row = 0
    col = 0
    for i in range(len(outputs)):
        plt.subplot(gs[i])
        plt.axis("off")
        if captions is not None:
            plt.title(captions[i])
        if hasattr(outputs, 'at'):
            out = outputs.at(i) if cpu else outputs.as_cpu().at(i)
        else:
            out = outputs[i]
        if len(out.shape) == 2 or (len(out.shape) == 3 and out.shape[-1] == 1):
            plt.imshow(out, cmap='gray')
        else:
            plt.imshow(out)

In [7]:
def as_param_(mag):
    return np.array(mag)

def as_param_data_node_(magnitudes, magnitude_idx, as_param):
    params = np.array([as_param(magnitude) for magnitude in magnitudes])
    params = types.Constant(params)
    print(params)
    return params[magnitude_idx]
    
def augmentation(lo, hi, num_magnitudes=11):
    magnitudes = types.Constant(np.linspace(lo, hi, num_magnitudes, dtype=np.float32))
    def decorator(function):
        @wraps(function)
        def wrapper(samples, magnitude_idx):
            magnitude = magnitudes[magnitude_idx]
            return function(samples, magnitude)  
        return wrapper
    return decorator

In [8]:
#level, magnitude, param
    

In [9]:
def warp_x_param(magnitude):
    return [magnitude, 0]

@augmentation(0, 0.3)
def shear_x(samples, parameter):
    
    mt = fn.transforms.shear(shear=parameter)
    return fn.warp_affine(samples, matrix=mt, fill_value=0, inverse_map=False)

def warp_y_param(magnitude):
    return [0, magnitude]

@augmentation(0, 0.3, as_param=warp_y_param)
def shear_y(samples, parameter):
    mt = fn.transforms.shear(shear=parameter)
    return fn.warp_affine(samples, matrix=mt, fill_value=0, inverse_map=False)

@augmentation(0, 250, as_param=warp_x_param)
def translate_x(samples, parameter):
    mt = fn.transforms.translation(offset=parameter)
    return fn.warp_affine(samples, matrix=mt, fill_value=0, inverse_map=False)

@augmentation(0, 250)
def translateY(samples, parameter, as_param=warp_y_param):
    mt = fn.transforms.translation(offset=parameter)
    return fn.warp_affine(samples, matrix=mt, fill_value=0, inverse_map=False)

@augmentation(0, 30)
def rotate(samples, parameter):
    return fn.rotate(samples, angle=parameter, fill_value=0)

@augmentation(0.1, 1.9)
def brightness(samples, parameter):
    return fn.brightness(samples, brightness=parameter)

@augmentation(0.1, 1.9)
def contrast(samples, parameter):
    return fn.contrast(samples, contrast=parameter)

@augmentation(0.1, 1.9)
def color(samples, parameter):
    return fn.saturation(samples, saturation=parameter)

@augmentation(0, 4)
def posterize(samples, parameter):
    nbits = np.round(parameter).astype(np.int32)
    mask = np.array(255 ^ (2 ** (8 - nbits) - 1), dtype=np.uint8)
    return samples & mask

@augmentation(0, 110)
def solarize(samples, threshold):
    samples_inv = 255 - samples
    mask_left = samples < np.uint8(threshold)
    mask_right = 1 - mask_left
    return fn.cast(mask_left * samples + mask_right * samples_inv, dtype=types.UINT8)

@augmentation(0, 256)
def solarize_add(samples, shift):
    samples_shifted = fn.cast_like(samples + shift, samples)
    mask_left = samples < 128
    mask_right = 1 - mask_left
    return fn.cast_like(mask_left * samples_shifted + mask_right * samples, samples)

@augmentation(0.1, 1.9)
def sharpness(samples, magnitude):
    blur = np.array([
        [1, 1, 1],
        [1, 5, 1],
        [1, 1, 1]],
        dtype=np.float32) / 13
    ident = np.array([
        [0, 0, 0],
        [0, 1, 0],
        [0, 0, 0]],
        dtype=np.float32)
    kernel = (1 - magnitude) * blur + magnitude * ident
    return fn.experimental.filter(samples, kernel)

@augmentation(0, 0)
def invert(samples, _):
    return fn.cast_like(255 - samples, samples)

@augmentation(0, 0)
def equalize(samples, _):
    return fn.experimental.equalize(samples)

@augmentation(0, 0)
def autocontrast(samples, _):
    lo, hi = fn.reductions.min(samples, axes=[-3, -2]), fn.reductions.max(samples, axes=[-3, -2])
    lo = fn.expand_dims(lo, axes=[0, 1])
    hi = fn.expand_dims(hi, axes=[0, 1])
    return fn.cast_like((samples - lo) * (255 / (hi - lo)), samples)

In [10]:
def get_operations(num_total_ops, N):
    rng = np.random.default_rng(12345)
    def inner(sample_info):
        return rng.choice(range(num_total_ops), N)
    return inner

def split_level(op_range_lo, op_range_hi, ops, samples, M, level_op_idxs):
    assert op_range_lo <= op_range_hi
    if op_range_lo == op_range_hi:
        return ops[op_range_lo](samples, M)
    mid = (op_range_lo + op_range_hi) // 2
    if level_op_idxs <= mid:
        return split_level(op_range_lo, mid, ops, samples, M, level_op_idxs)
    else:
        return split_level(mid + 1, op_range_hi, ops, samples, M, level_op_idxs)

def trivial_augment(ops, samples, num_bins=11):  # todo should be 31
    num_total_ops = len(ops)
    op_idxs = fn.external_source(source=get_operations(num_total_ops, 1), batch=False)
    magnitudes = fn.random.uniform(range=[0, num_bins - 1], dtype=types.INT32)
    samples = split_level(0, num_total_ops - 1, ops, samples, magnitudes, op_idxs[0])
    return samples

In [11]:
# trivial_aug_ops = [shearX, shearY, translateX, translateY, rotate, brightness, contrast, color, posterize, solarize, solarize_add, sharpness, invert, equalize, autocontrast]
trivial_aug_ops = [shear_x, shear_y]


@experimental.pipeline_def(enable_conditionals=True, seed=42)
def pipeline():
    images, _ = fn.readers.file(name="Reader", file_root=data_path, random_shuffle=True, seed=42)
    shapes = fn.peek_image_shape(images)
    images = fn.decoders.image(images, device="mixed")
    images = fn.resize(images, resize_x=400, resize_y=400)
    return trivial_augment(trivial_aug_ops, images), shapes

In [12]:
# @pipeline_def
# def pipeline():
#     a = types.Constant(np.array([2, 4, 6, 8, 10, 12, 14, 16, 18, 20, 22], dtype=np.int32))
#     selected = fn.random.uniform(range=[0, 11], dtype=types.INT32)
#     return selected, a[selected]

In [13]:
p = pipeline(batch_size=12, device_id=0, num_threads=4)

DataNode(name="__Split_9[0]", device="cpu")
DataNode(name="__Split_12[0]", device="cpu")
DataNode(name="__Split_9[1]", device="cpu")
DataNode(name="__Split_20[1]", device="cpu")


In [16]:
p.build()
p.save_graph_to_dot_file("hmhmhm.dot")

In [15]:
out, shapes = p.run()

RuntimeError: Critical error in pipeline:
Error when executing CPU operator conditional__Split, instance name: "__Split_23", encountered:
[/home/ktokarski/DALI/dali/pipeline/operator/builtin/split.cc:29] Assert on "input.num_samples() == predicate.num_samples()" failed: Split description must cover whole input, got 7 input samples and 12 elements denoting the split.
Stacktrace (11 entries):
[frame 0]: /home/ktokarski/DALI/build/dali/python/nvidia/dali/libdali.so(+0xca832) [0x7f0ff2f27832]
[frame 1]: /home/ktokarski/DALI/build/dali/python/nvidia/dali/libdali.so(+0x198672) [0x7f0ff2ff5672]
[frame 2]: /home/ktokarski/DALI/build/dali/python/nvidia/dali/libdali.so(+0x16df9b) [0x7f0ff2fcaf9b]
[frame 3]: /home/ktokarski/DALI/build/dali/python/nvidia/dali/libdali.so(dali::Executor<dali::AOT_WS_Policy<dali::UniformQueuePolicy>, dali::UniformQueuePolicy>::RunHelper(dali::OpNode&, dali::Workspace&, unsigned long)+0x5d1) [0x7f0ff2fa24f1]
[frame 4]: /home/ktokarski/DALI/build/dali/python/nvidia/dal

In [ ]:
out = [np.array(s) for s in out.as_cpu()]

In [ ]:
display_imgs(out, columns=4)